In [ ]:
import numpy as np
from numpy import random
BASE_CHANCE: np.ndarray = np.array([0.5]+[0.45]*2+[0.4]*3+[0.35]*4+[0.3]*10)
TARGET_LEVELS: np.ndarray = np.arange(21)
def roll(threshold: float)->int:
    assert 0 <= threshold , "Threshold must be non-negative"
    return int(random.random() < threshold)

In [ ]:
base_level: int = 4
max_level: int = 9
amplify_percent: float = 4.83
num_trials: int = 5000
single_enhance_cost: int = 34625
single_item_cost: int = 1_100_000
protection_min_level: int = 5
assert max_level <= 20, "Max level must be less than or equal to 20"
assert protection_min_level >= 2, "WTF bro why are you protecting nothing?"
protection_cost: int = single_item_cost // single_enhance_cost
print(f"Protection cost: {protection_cost}")

In [ ]:


from torch import threshold


arr_times_taken: np.ndarray = np.zeros((21, num_trials), dtype=int)
chance: float = 0.0
level: int = 0
success: bool = False
for current_level in TARGET_LEVELS[2:8]:
    for current_trial in range(num_trials):
        level = 0
        while level < current_level:
            arr_times_taken[current_level][current_trial] += 1
            chance = BASE_CHANCE[level] * (1 + amplify_percent/100)
            success = roll(chance)
            level += success + roll(0.01)
            level *= success

    # sanity check
    print(f"Level {current_level} completed, average {arr_times_taken[current_level].mean():.2f} rolls per trial.")

In [ ]:
arr_times_taken_protected: np.ndarray = np.zeros((21, num_trials), dtype=int)
arr_protected_count: np.ndarray = np.zeros((21, num_trials), dtype=int)
chance_protected: float = 0.0
level_protected: int = base_level
success_protected: bool = False
for current_level in TARGET_LEVELS[base_level+1:max_level+1]:
    for current_trial in range(num_trials):
        level_protected = base_level
        while level_protected < current_level:
            arr_times_taken_protected[current_level][current_trial] += 1
            chance_protected = BASE_CHANCE[level_protected] * (1 + amplify_percent/100)
            success_protected = roll(chance_protected)
            if (level_protected >= protection_min_level)  and (not success_protected):
                level_protected -= 1
                arr_times_taken_protected[current_level][current_trial] += protection_cost
                arr_protected_count[current_level][current_trial] += 1
                continue
            level_protected += success_protected # means nothing if failed, +1 if succeeded
            level_protected += roll(0.01) # 1% chance to +2 instead of +1 due to blessed tea
            level_protected *= success_protected
            
    # sanity check
    print(f"Level {current_level} completed, average {arr_times_taken_protected[current_level].mean():.2f} rolls per trial.")

In [ ]:
# display median and stdev for each level in tabular format
print("Level\tProtected Median\tProtected Stdev\tProtected Count")
for i in TARGET_LEVELS[:max_level+1]:
    protected_median: float = np.median(arr_times_taken_protected[i])
    protected_stdev: float = np.std(arr_times_taken_protected[i])
    protected_count_median: int = np.median(arr_protected_count[i])
    protected_count_stdev: float = np.std(arr_protected_count[i])
    print(f"{i}\t{protected_median:.0f}\t{protected_stdev:.2f}\t{protected_count_median:.0f}±{protected_count_stdev:.2f}")

In [ ]:
# display mean, stdev, and percentiles for a specific level
focus_level: int = 7

protected_mean: float = np.mean(arr_times_taken_protected[focus_level])
protected_stdev: float = np.std(arr_times_taken_protected[focus_level])
protected_percentiles: np.ndarray = np.percentile(arr_times_taken_protected[focus_level], [80, 90, 95, 98, 99, 99.5])
print(f"\nFocus level: {base_level} -> {focus_level} (Protected)")
print(f"Mean: {protected_mean:.2f}")
print(f"Stdev: {protected_stdev:.2f}")
print(f"80% percentile: {protected_percentiles[0]:.0f}")
print(f"90% percentile: {protected_percentiles[1]:.0f}")
print(f"95% percentile: {protected_percentiles[2]:.0f}")
print(f"98% percentile: {protected_percentiles[3]:.0f}")
print(f"99% percentile: {protected_percentiles[4]:.0f}")
print(f"99.5% percentile: {protected_percentiles[5]:.0f}")

protection_mean: float = np.mean(arr_protected_count[focus_level])
protection_stdev: float = np.std(arr_protected_count[focus_level])
protection_percentiles: np.ndarray = np.percentile(arr_protected_count[focus_level], [80, 90, 95, 98, 99, 99.5])
print(f"\nFocus level: {base_level} -> {focus_level} (Protection Count)")
print(f"Mean: {protection_mean:.2f}")
print(f"Stdev: {protection_stdev:.2f}")
print(f"80% percentile: {protection_percentiles[0]:.0f}")
print(f"90% percentile: {protection_percentiles[1]:.0f}")
print(f"95% percentile: {protection_percentiles[2]:.0f}")
print(f"98% percentile: {protection_percentiles[3]:.0f}")
print(f"99% percentile: {protection_percentiles[4]:.0f}")
print(f"99.5% percentile: {protection_percentiles[5]:.0f}") 